# Step 2: Agents in a Workflow (Non-Streaming)

## Overview

This notebook demonstrates how to integrate AI agents into workflows using the Agent Framework. We'll create a two-agent workflow where:
1. **Writer Agent** - Creates or edits content
2. **Reviewer Agent** - Evaluates and provides feedback

### Key Concepts:

- **Agents as Workflow Executors**: Wrap chat agents created by `AzureAIAgentClient` inside workflow executors
- **Automatic Output Yielding**: Agents automatically yield outputs when they complete
- **Non-Streaming Execution**: Using `workflow.run()` for synchronous results
- **Agent Events**: Capturing and displaying agent run events

### Prerequisites:

- ✅ Microsoft Foundry Project configured with required environment variables
- ✅ Azure CLI authentication (`az login` completed)
- ✅ Basic familiarity with WorkflowBuilder, executors, and edges

## Import Required Libraries

In [ ]:
import asyncio

from agent_framework import AgentRunEvent, WorkflowBuilder, ChatMessage, Executor, WorkflowContext, handler
from azure.identity.aio import AzureCliCredential
from agent_framework.azure import AzureAIAgentClient
from pathlib import Path  # For working with file paths
import os  # For environment variables
import time  # For sleep function
from dotenv import load_dotenv  # For loading environment variables from .env file
# Get the path to the .env file which is in the parent directory
notebook_path = Path().absolute()  # Get absolute path of current notebook
parent_dir = notebook_path.parent  # Get parent directory
load_dotenv('../../.env')  # Load environment variables from .env file

## Create Azure AI Agent Client

We'll use **AzureAIAgentClient** with `AzureCliCredential` for authentication.

### Authentication:
`AzureCliCredential` uses Azure CLI authentication (`az login`).

In [ ]:
print("✅ Environment variables loaded")

In [ ]:
# Create the Azure AI credential
credential = AzureCliCredential()

print("✅ Azure AI credential created successfully!")

## Define the Writer Agent

The Writer Agent is responsible for creating and editing content based on requirements and feedback.

**Agent Configuration:**
- **Instructions**: Guides the agent's behavior and role
- **Name**: Identifier for the agent (used in events and debugging)

In [ ]:
class Writer(Executor):
    """Writer executor that wraps an Azure AI agent."""
    
    def __init__(self, client: AzureAIAgentClient, id: str = "writer"):
        self.agent = client.create_agent(
            instructions=(
                "You are an excellent content writer. You create new content and edit contents based on the feedback."
            ),
            name=id,
        )
        super().__init__(id=id)
    
    @handler
    async def handle(self, message: ChatMessage, ctx: WorkflowContext[list[ChatMessage]]) -> None:
        """Generate content and forward messages to next executor."""
        # Convert single message to list
        messages = [message]
        response = await self.agent.run(messages)
        
        # Print the writer's response
        print(f"\n✍️ Writer Generated: {response.text}\n")
        
        full_conversation = messages + list(response.messages)
        await ctx.send_message(full_conversation)
    
    @handler
    async def handle_list(self, messages: list[ChatMessage], ctx: WorkflowContext[list[ChatMessage]]) -> None:
        """Handle feedback from reviewer - revise content based on feedback."""
        response = await self.agent.run(messages)
        
        # Print the writer's revised response
        print(f"\n✍️ Writer Revised: {response.text}\n")
        
        full_conversation = messages + list(response.messages)
        await ctx.send_message(full_conversation)

print("✅ Writer executor class defined")

## Define the Reviewer Agent

The Reviewer Agent evaluates content and provides actionable feedback.

**Key Characteristics:**
- Provides concise, actionable feedback
- Evaluates content quality
- Finalizes the result

In [ ]:
class Reviewer(Executor):
    """Reviewer executor that wraps an Azure AI agent and yields final output."""
    
    def __init__(self, client: AzureAIAgentClient, id: str = "reviewer", max_iterations: int = 2):
        self.agent = client.create_agent(
            instructions=(
                "You are an excellent content reviewer."
                "Provide actionable feedback to the writer about the provided content, that the writer can use to improve it."
                "Provide the feedback in the most concise manner possible."  
                "IMPORTANT: Never approve the first draft. Always provide constructive feedback on the first iteration."
                "After seeing revisions, respond with 'APPROVED: ' followed by the final content."
            ),
            name=id,
        )
        self.max_iterations = max_iterations
        self.iteration_count = 0
        super().__init__(id=id)
    
    @handler
    async def handle(self, messages: list[ChatMessage], ctx: WorkflowContext[list[ChatMessage], str]) -> None:
        """Review content and either send feedback to writer or yield final output."""
        self.iteration_count += 1
        response = await self.agent.run(messages)
        
        # Print the reviewer's feedback
        print(f"\n📝 Reviewer Feedback (Iteration {self.iteration_count}): {response.text}\n")

        # Never approve on first iteration, check approval keyword or max iterations
        if self.iteration_count > 1 and response.text.startswith("APPROVED:"):
            print(f"🎯 Workflow complete after {self.iteration_count} iteration(s)\n")
            await ctx.yield_output(response.text)
        elif self.iteration_count >= self.max_iterations:
            print(f"🎯 Max iterations ({self.max_iterations}) reached\n")
            await ctx.yield_output(response.text)
        else:
            # Send feedback back to writer for revision
            full_conversation = messages + list(response.messages)
            await ctx.send_message(full_conversation)

## Build the Workflow

### Workflow Structure:

```
Writer Agent → Reviewer Agent
      ↑            |
      └────────────┘
    (feedback loop until approved)
```

Using the fluent `WorkflowBuilder` API:

1. Set the writer as the start node. Build the workflow

2. Connect an edge from writer to reviewer. Add feedback edge from reviewer back to writer (with termination condition)

In [ ]:
# Instantiate the agent-backed executors
writer = Writer(AzureAIAgentClient(async_credential=credential))
reviewer = Reviewer(AzureAIAgentClient(async_credential=credential))

# Build the workflow with feedback loop
workflow = (
    WorkflowBuilder()
    .set_start_executor(writer)
    .add_edge(writer, reviewer)      # Writer sends to Reviewer
    .add_edge(reviewer, writer)      # Reviewer sends feedback back to Writer
    .build()
)

print("✅ Workflow built successfully!")
print("   Writer → Reviewer → Writer (feedback loop)")

## Run the Workflow

### Execution Flow:

1. User provides initial message
2. Writer Agent creates content based on the request
3. Writer's output is automatically sent to Reviewer Agent
4. Reviewer Agent evaluates and provides feedback
5. Workflow completes when all agents are done

### Event Handling:

- `AgentRunEvent` - Captures agent responses
- `get_outputs()` - Retrieves final workflow outputs
- `get_final_state()` - Shows workflow completion status

In [ ]:
# Run the workflow with a user message
user_message = "Create a 100 word description for a new electric SUV that is affordable and fun to drive."

print(f"\n📝 User Request: {user_message}\n")
print("=" * 60)

# Execute the workflow
events = await workflow.run(ChatMessage(role="user", text=user_message))

# Print all events to see what's available
print("\n🔍 All Events:")
for i, event in enumerate(events):
    print(f"{i}. {type(event).__name__}: {event}")

# Print agent run events (if any)
print("\n🤖 Agent Run Events:")
agent_events = [e for e in events if isinstance(e, AgentRunEvent)]
if agent_events:
    for event in agent_events:
        print(f"\n--- {event.executor_id.upper()} OUTPUT ---")
        print(event.data)
else:
    print("No AgentRunEvent found (Azure AI Agents don't emit these by default)")

# Print final outputs
print(f"\n{'=' * 60}")
print("\n📤 FINAL WORKFLOW OUTPUT:")
print("=" * 60)
for output in events.get_outputs():
    print(output)

# Print final state
print(f"\n{'=' * 60}")
print(f"✅ Final state: {events.get_final_state()}")
print("=" * 60)

## Expected Output

### Sample Execution:

```
writer: "Charge Up Your Adventure—Affordable Fun, Electrified!"

reviewer: Slogan: "Plug Into Fun—Affordable Adventure, Electrified."

**Feedback:**
- Clear focus on affordability and enjoyment.
- "Plug into fun" connects emotionally and highlights electric nature.
- Consider specifying "SUV" for clarity in some uses.
- Strong, upbeat tone suitable for marketing.

============================================================
Workflow Outputs: ['Slogan: "Plug Into Fun—Affordable Adventure, Electrified."

**Feedback:**
- Clear focus on affordability and enjoyment.
- "Plug into fun" connects emotionally and highlights electric nature.
- Consider specifying "SUV" for clarity in some uses.
- Strong, upbeat tone suitable for marketing.']

Final state: WorkflowRunState.COMPLETED
```

## Key Takeaways

### Agent Integration

✅ **Agents as Executors**
- AI agents created with `AzureAIAgentClient` can be used directly in workflows
- No need for custom executor wrappers
- Seamless integration with WorkflowBuilder

✅ **Automatic Output Handling**
- Agents automatically yield outputs when they complete
- No explicit `ctx.yield_output()` needed
- Simplifies workflow construction

✅ **Event-Driven Architecture**
- `AgentRunEvent` captures agent responses
- Events can be filtered and processed
- Full visibility into workflow execution

### Non-Streaming vs Streaming

**Non-Streaming (`workflow.run()`):**
- Waits for complete responses
- Returns all events at once
- Simpler for batch processing
- Used in this example

**Streaming (`workflow.run_stream()`):**
- Real-time response chunks
- Better user experience for long responses
- Covered in Step 3

### Workflow Pattern

This example demonstrates a **Sequential Agent Chain**:
```
User Input → Writer Agent → Reviewer Agent → Final Output
```

Common use cases:
- Content creation and review
- Multi-stage processing pipelines
- Quality assurance workflows

### Next Steps

Continue to **Step 3** to learn about streaming responses and real-time output!